# 01 — Pretrain Hybrid CNN + Transformer on ARC-AGI-2 (Colab Pro)

Runtime → Change runtime type → A100 / L4 (recommended) or T4 (slow but works).
Persists checkpoints to `/content/drive/MyDrive/arc_hybrid_runs/` so a session reset doesn't lose work.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pathlib

REPO_DIR = pathlib.Path('/content/arc_hybrid_repo')
if not REPO_DIR.exists():
    !git clone https://github.com/Nitish05/ARC-AGI.git {REPO_DIR}

%cd /content/arc_hybrid_repo
!pip install -q -r requirements.txt

In [ ]:
# Pull latest from GitHub (run this whenever you've pushed local edits) and
# enable hot-reload so edits to arc_hybrid/*.py take effect on the next cell run
# without needing a kernel restart.
!git -C /content/arc_hybrid_repo pull --ff-only
%load_ext autoreload
%autoreload 2

In [ ]:
# Download datasets into data/raw/
!mkdir -p data/raw
!git clone --depth 1 https://github.com/arcprize/ARC-AGI-2 data/raw/ARC-AGI-2 || true
!git clone --depth 1 https://github.com/fchollet/ARC-AGI data/raw/ARC-AGI || true
# RE-ARC (optional, large): uncomment the next line to include
# !git clone --depth 1 https://github.com/michaelhodel/re-arc data/raw/re-arc

In [ ]:
# Smoke test: 100 steps, tiny config
!python -m arc_hybrid.train.pretrain --config configs/pretrain_small.yaml --steps 100

In [ ]:
# Full pretrain — point output dir at Drive so checkpoints survive a session reset.
import yaml, pathlib
cfg_path = pathlib.Path('configs/pretrain_medium.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
cfg['logging']['out_dir'] = '/content/drive/MyDrive/arc_hybrid_runs/medium_v1'
tmp = pathlib.Path('configs/_active.yaml')
tmp.write_text(yaml.safe_dump(cfg))
!python -m arc_hybrid.train.pretrain --config configs/_active.yaml